In [2]:
import numpy as np

corpus = [
    "low",
    "lower",
    "lowest",
    "new",
    "newer",
    "widest"
]

corpus = [word.lower().strip() for word in corpus]

print("========== CORPUS ==========")
print(corpus)

tokens = []

for word in corpus:
    chars = list(word)
    chars.append("</w>")
    tokens.extend(chars)

print("\n========== TOKENS ==========")
print(tokens)

vocabulary = sorted(list(set(tokens)))

print("\n========== VOCABULARY ==========")
print(vocabulary)

vocab_size = len(vocabulary)

print("\nVocabulary Size :", vocab_size)


token_to_id = {}
id_to_token = {}

for index, token in enumerate(vocabulary):
    token_to_id[token] = index
    id_to_token[index] = token

print("\n========== TOKEN IDS ==========")

for token in vocabulary:
    print(token, "->", token_to_id[token])

token_ids = []

for token in tokens:
    token_ids.append(token_to_id[token])

print("\n========== TOKEN ID SEQUENCE ==========")
print(token_ids)


inputs = np.array(token_ids[:-1])
targets = np.array(token_ids[1:])

print("\nInputs Shape :", inputs.shape)
print("Targets Shape :", targets.shape)

embedding_dim = 16

embedding_matrix = np.random.randn(vocab_size, embedding_dim)

embedded_inputs = embedding_matrix[inputs]

print("\nEmbedding Matrix Shape :", embedding_matrix.shape)
print("Embedded Input Shape :", embedded_inputs.shape)

sequence_length = embedded_inputs.shape[0]

position_encoding = np.zeros((sequence_length, embedding_dim))

for pos in range(sequence_length):

    for i in range(embedding_dim):

        angle = pos / np.power(10000, (2 * (i // 2)) / embedding_dim)

        if i % 2 == 0:
            position_encoding[pos][i] = np.sin(angle)
        else:
            position_encoding[pos][i] = np.cos(angle)

embedded_inputs = embedded_inputs + position_encoding

print("\nFinal Input Shape :", embedded_inputs.shape)




========== CORPUS ==========
['low', 'lower', 'lowest', 'new', 'newer', 'widest']

========== TOKENS ==========
['l', 'o', 'w', '</w>', 'l', 'o', 'w', 'e', 'r', '</w>', 'l', 'o', 'w', 'e', 's', 't', '</w>', 'n', 'e', 'w', '</w>', 'n', 'e', 'w', 'e', 'r', '</w>', 'w', 'i', 'd', 'e', 's', 't', '</w>']

========== VOCABULARY ==========
['</w>', 'd', 'e', 'i', 'l', 'n', 'o', 'r', 's', 't', 'w']

Vocabulary Size : 11

========== TOKEN IDS ==========
</w> -> 0
d -> 1
e -> 2
i -> 3
l -> 4
n -> 5
o -> 6
r -> 7
s -> 8
t -> 9
w -> 10

========== TOKEN ID SEQUENCE ==========
[4, 6, 10, 0, 4, 6, 10, 2, 7, 0, 4, 6, 10, 2, 8, 9, 0, 5, 2, 10, 0, 5, 2, 10, 2, 7, 0, 10, 3, 1, 2, 8, 9, 0]

Inputs Shape : (33,)
Targets Shape : (33,)

Embedding Matrix Shape : (11, 16)
Embedded Input Shape : (33, 16)

Final Input Shape : (33, 16)


In [3]:


Wq = np.random.randn(embedding_dim, embedding_dim)
Wk = np.random.randn(embedding_dim, embedding_dim)
Wv = np.random.randn(embedding_dim, embedding_dim)

Q = embedded_inputs @ Wq
K = embedded_inputs @ Wk
V = embedded_inputs @ Wv

print("Query Shape :", Q.shape)
print("Key Shape   :", K.shape)
print("Value Shape :", V.shape)


scores = (Q @ K.T) / np.sqrt(embedding_dim)

print("\nAttention Score Shape :", scores.shape)


mask = np.triu(
    np.ones((sequence_length, sequence_length)),
    k=1
)

scores = np.where(mask == 1, -1e9, scores)

print("Causal Mask Applied")


def softmax(x):

    x = x - np.max(x, axis=1, keepdims=True)

    exp = np.exp(x)

    return exp / np.sum(exp, axis=1, keepdims=True)

attention_weights = softmax(scores)

print("\nAttention Weight Shape :", attention_weights.shape)


context = attention_weights @ V

print("Context Shape :", context.shape)


Wo = np.random.randn(
    embedding_dim,
    vocab_size
)

bias = np.zeros(vocab_size)

logits = context @ Wo + bias

print("\nLogits Shape :", logits.shape)

probabilities = softmax(logits)

print("Probability Shape :", probabilities.shape)

prediction = np.argmax(probabilities[0])

print("\nFirst Predicted Token :")

print(id_to_token[prediction])



Query Shape : (33, 16)
Key Shape   : (33, 16)
Value Shape : (33, 16)

Attention Score Shape : (33, 33)
Causal Mask Applied

Attention Weight Shape : (33, 33)
Context Shape : (33, 16)

Logits Shape : (33, 11)
Probability Shape : (33, 11)

First Predicted Token :
w


In [4]:


def cross_entropy(predictions, targets):

    loss = 0

    for i in range(len(targets)):

        p = predictions[i][targets[i]]

        p = max(p, 1e-10)

        loss -= np.log(p)

    return loss / len(targets)


loss = cross_entropy(probabilities, targets)

print("\nInitial Loss :", loss)


learning_rate = 0.01
epochs = 100

for epoch in range(epochs):

    # Forward

    Q = embedded_inputs @ Wq
    K = embedded_inputs @ Wk
    V = embedded_inputs @ Wv

    scores = (Q @ K.T) / np.sqrt(embedding_dim)
    scores = np.where(mask == 1, -1e9, scores)

    attention = softmax(scores)

    context = attention @ V

    logits = context @ Wo + bias

    probabilities = softmax(logits)

    loss = cross_entropy(probabilities, targets)

    # Gradient

    grad = probabilities.copy()

    for i in range(len(targets)):
        grad[i][targets[i]] -= 1

    grad = grad / len(targets)

    dWo = context.T @ grad
    db = np.sum(grad, axis=0)

    Wo -= learning_rate * dWo
    bias -= learning_rate * db

    if (epoch + 1) % 10 == 0:
        print("Epoch", epoch + 1, " Loss =", loss)


print("\nTraining Completed")


def generate(start_token, steps=10):

    generated = [start_token]

    current = start_token

    for _ in range(steps):

        if current not in token_to_id:
            break

        idx = token_to_id[current]

        x = embedding_matrix[idx].reshape(1, embedding_dim)

        q = x @ Wq
        k = x @ Wk
        v = x @ Wv

        score = (q @ k.T) / np.sqrt(embedding_dim)

        weight = softmax(score)

        context = weight @ v

        out = context @ Wo + bias

        prob = softmax(out)

        next_id = np.argmax(prob)

        next_token = id_to_token[next_id]

        generated.append(next_token)

        current = next_token

        if next_token == "</w>":
            break

    return generated


print("\n==============================")
print("AUTOREGRESSIVE TEXT GENERATION")
print("==============================")

result = generate("l")

print("Generated Tokens :")
print(result)

word = ""

for token in result:

    if token != "</w>":

        word += token

print("\nGenerated Word :", word)

print("\n==============================")
print("MODEL SUMMARY")
print("==============================")

print("Vocabulary Size :", vocab_size)
print("Embedding Dimension :", embedding_dim)
print("Sequence Length :", sequence_length)
print("Epochs :", epochs)
print("Final Loss :", loss)

print("\nAutoregressive Model Completed Successfully.")


Initial Loss : 19.888593903867303
Epoch 10  Loss = 17.37145626087433
Epoch 20  Loss = 15.229364132319633
Epoch 30  Loss = 13.351416284127637
Epoch 40  Loss = 12.457596235817297
Epoch 50  Loss = 11.612505808149743
Epoch 60  Loss = 10.409523324523397
Epoch 70  Loss = 9.122696183439578
Epoch 80  Loss = 8.03402976424965
Epoch 90  Loss = 7.101688178463208
Epoch 100  Loss = 6.328052455146057

Training Completed

AUTOREGRESSIVE TEXT GENERATION
Generated Tokens :
['l', 'w', 'd', 'o', 'o', 'o', 'o', 'o', 'o', 'o', 'o']

Generated Word : lwdoooooooo

MODEL SUMMARY
Vocabulary Size : 11
Embedding Dimension : 16
Sequence Length : 33
Epochs : 100
Final Loss : 6.328052455146057

Autoregressive Model Completed Successfully.
